# step 3 — pooled union: compilation & inspection

Compiles the union of System 1 (cosine similarity) and System 2 (BERTopic) candidates,
saves `_pooled_union.json`, and provides cells for reviewing the candidate set.

**Union rationale:** the union (not intersection) maximises recall — both systems have
independent failure modes, and any document missed by one may be caught by the other.

| section | what it shows |
|---|---|
| §3 | summary counts by source system + year distribution |
| §4 | high-confidence docs found by both systems |
| §5 | docs found only by cosine similarity |
| §5a | **decision:** which System-1-only policies to include (index list) |
| §6 | docs found only by BERTopic — inspect topic by topic |
| §6a | **decision:** which BERTopic topics to keep |
| §7 | noise pool spot-check (topic -1) |
| §8 | compile final policy set & save |
| §9 | export evaluation log (Markdown) |

In [9]:
import gzip
import json
import random
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', None)

# ── CONFIGURATION ─────────────────────────────────────────────────────────────

def _find_root(marker='CLAUDE.md'):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError('project root not found — run from within the project directory')

PROJECT_ROOT      = _find_root()
OUTPUT_PREFIX     = 'germany_2008-2015_2019-2022'
BASE              = PROJECT_ROOT / 'data' / 'processed' / 'germany'

S1_FILE           = BASE / f'{OUTPUT_PREFIX}_cosinesim_final.json' #selection from system 1, cosine similarity
S2_FILE           = BASE / f'{OUTPUT_PREFIX}_berttopic_candidates.json' #policies that are in social policy relevant topics (from triage)
NOISE_FILE        = BASE / f'{OUTPUT_PREFIX}_berttopic_noise_pool.json' #policies that could not be matched to any topic  at all
TOPIC_ASSIGN_FILE = BASE / f'{OUTPUT_PREFIX}_berttopic_topic_assignments.json.gz' #all policies with their assigned topics
TOPIC_INFO_FILE   = BASE / f'{OUTPUT_PREFIX}_berttopic_topic_info.json'
OUT_FILE          = BASE / f'{OUTPUT_PREFIX}_pooled_union.json'

## §1. Load data

In [10]:
s1_docs    = json.loads(S1_FILE.read_text(encoding='utf-8'))
s2_docs    = json.loads(S2_FILE.read_text(encoding='utf-8'))
noise_docs = json.loads(NOISE_FILE.read_text(encoding='utf-8')) if NOISE_FILE.exists() else []
topic_info = json.loads(TOPIC_INFO_FILE.read_text(encoding='utf-8')) if TOPIC_INFO_FILE.exists() else []

if TOPIC_ASSIGN_FILE.exists():
    with gzip.open(TOPIC_ASSIGN_FILE, 'rt', encoding='utf-8') as _fh:
        topic_assign = json.load(_fh)
else:
    topic_assign = []

print(f'System 1 (cosine similarity):   {len(s1_docs):>4} candidates')
print(f'System 2 (BERTopic):            {len(s2_docs):>4} candidates')
print(f'System 2 Noise pool (topic -1): {len(noise_docs):>4} docs')

System 1 (cosine similarity):    593 candidates
System 2 (BERTopic):             892 candidates
System 2 Noise pool (topic -1):  791 docs


## §2. Compile union & save

In [4]:
s1_by_title = {d['title'].strip(): d for d in s1_docs}
s2_by_title = {d['title'].strip(): d for d in s2_docs}
all_titles  = sorted(set(s1_by_title) | set(s2_by_title))

union  = []
counts = {'both': 0, 's1_only': 0, 's2_only': 0}

for title in all_titles:
    in_s1  = title in s1_by_title
    in_s2  = title in s2_by_title
    base   = s1_by_title[title] if in_s1 else s2_by_title[title]
    source = 'both' if (in_s1 and in_s2) else ('s1_only' if in_s1 else 's2_only')
    union.append({**base, 'system_sources': source})
    counts[source] += 1

union.sort(key=lambda x: x.get('date_published', ''))
OUT_FILE.write_text(json.dumps(union, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved {len(union)} candidates to {OUT_FILE.name}')

df_union = pd.DataFrame(union)
df_union['year'] = pd.to_datetime(df_union['date_published'], errors='coerce').dt.year

Saved 1059 candidates to germany_2008-2015_2019-2022_pooled_union.json


## §3. Overview

In [19]:
n_both  = counts['both']
n_s1    = counts['s1_only']
n_s2    = counts['s2_only']
n_total = len(union)

pd.DataFrame({
    'n': [n_both, n_s1, n_s2, n_total],
    'share': [f'{n_both/n_total:.0%}', f'{n_s1/n_total:.0%}', f'{n_s2/n_total:.0%}', '100%'],
}, index=['both systems (high confidence)', 'System 1 only (cosine sim)', 'System 2 only (BERTopic)', 'total'])

,n,share
both systems (high confidence),345,33%
System 1 only (cosine sim),212,20%
System 2 only (BERTopic),502,47%
total,1059,100%


In [ ]:
pd.crosstab(df_union['year'], df_union['system_sources'], margins=True)

## §4. High-confidence candidates (both systems)

Found by both cosine similarity and BERTopic — the strongest candidates.

In [ ]:
df_both = df_union[df_union['system_sources'] == 'both'][['title', 'date_published', 'year']].reset_index(drop=True)
print(f'{len(df_both)} docs found by both systems')
df_both

## §5. System 1 only (cosine similarity)

Found by cosine similarity but not by BERTopic — vocabulary close to seed descriptions
but did not cluster with other welfare-policy laws.

In [5]:
df_s1 = df_union[df_union['system_sources'] == 's1_only'][['title', 'date_published', 'year']].reset_index(drop=True)
print(f'{len(df_s1)} docs found only by cosine similarity')
df_s1

212 docs found only by cosine similarity


,title,date_published,year
0,Verordnung zur Übertragung der Befugnis zum Erlass von Rechtsverordnungen nach dem Dritten Buch Sozialgesetzbuch auf...,2008-01-25T00:00:00Z,2008
1,Verordnung über die Berufsausbildung zum Personaldienstleistungskaufmann/zur Personaldienstleistungskauffrau,2008-02-29T00:00:00Z,2008
2,Bekanntmachung nach § 141 Abs. 11 Satz 2 des Arzneimittelgesetzes,2008-03-31T00:00:00Z,2008
3,Verordnung zur Änderung der Arzneimittel- und Wirkstoffherstellungsverordnung,2008-04-04T00:00:00Z,2008
4,Verordnung über die Mindestbeitragsrückerstattung in der Lebensversicherung (Mindestzuführungsverordnung),2008-04-11T00:00:00Z,2008
5,Verordnung zur Vermeidung unbilliger Härten durch Inanspruchnahme einer vorgezogenen Altersrente (Unbilligkeitsveror...,2008-04-17T00:00:00Z,2008
6,Hinweis:Bundesgesetzblatt Teil II Nr. 10,2008-05-26T00:00:00Z,2008
7,"Verordnung über die Zuweisungen an das Sondervermögen ""Versorgungsfonds der Bundesagentur für Arbeit"" ( VFBAZV)",2008-06-16T00:00:00Z,2008
8,Fünfte Verordnung zur Änderung der Arzneimittelverschreibungsverordnung,2008-06-25T00:00:00Z,2008
9,Erstes Gesetz zur Änderung des Jugendschutzgesetzes,2008-06-30T00:00:00Z,2008


### §5a. Decision — System 1 only

Review the table above (the left-most column is the 0-based row index).  
Set `S1_INCLUDED_INDICES` in the cell below to the indices you want to **include**, then run **§8**.  
Leave as `[]` to exclude all System-1-only policies.

In [6]:
# ── DECISION: System 1 only ───────────────────────────────────────────────────
# Row indices (0-based, from the §5 table above) to include in the final set.
# Example: S1_INCLUDED_INDICES = [0, 3, 7]
# Leave as [] to exclude all System-1-only policies.

S1_INCLUDED_INDICES = [0, 5, 7, 11, 13, 14, 15, 23, 24, 25 ,26, 31, 33, 35, 38, 40, 46, 51, 53, 56, 61, 62, 63, 70, 74, 75, 80, 82, 83, 89, 96, 
                       97, 100, 102, 104, 105, 107, 114, 117, 119, 123, 125, 126, 128, 129, 130, 133, 137, 140, 141, 144, 145, 148, 150, 151, 
                       152, 153, 155, 158, 160, 161, 163, 165, 166, 168, 168, 170, 171, 172, 173, 174, 176, 178, 180, 181, 191, 195, 198, 205, 206, 
                       207, 208, 210 , 211]   # ← edit this list

## §6. System 2 only (BERTopic)

Found by BERTopic but not by cosine similarity — likely welfare-policy laws with
unusual vocabulary that did not match the seed descriptions. Inspect topic by topic.

Topic metadata is joined from `_berttopic_topic_assignments.json` since the
candidates file contains only raw documents without topic labels.

In [7]:
title_to_topic = {entry['doc']['title'].strip(): entry['topic_id'] for entry in topic_assign}
topic_to_words = {t['topic_id']: t['top_words'] for t in topic_info}

df_s2 = df_union[df_union['system_sources'] == 's2_only'].copy()
df_s2['berttopic_topic'] = df_s2['title'].str.strip().map(title_to_topic)
print(f'{len(df_s2)} docs found only by BERTopic\n')

for topic_id, group in df_s2.groupby('berttopic_topic'):
    words = topic_to_words.get(topic_id, [])
    top_words_str = ', '.join(words[:6])
    print(f'── topic {int(topic_id)}  (n={len(group)})  top words: {top_words_str}')
    display(group[['title', 'date_published']].reset_index(drop=True))
    print()

502 docs found only by BERTopic

── topic 7  (n=42)  top words: meisterprfung, meisterprfung ii, ii, fortbildungsabschluss, anerkannt fortbildungsabschluss, professional


,title,date_published
0,Verordnung über das Meisterprüfungsberufsbild und über die Prüfungsanforderungen in den Teilen I und II der Meisterp...,2008-03-18T00:00:00Z
1,Verordnung über die Berufsausbildung zum Seiler und zur Seilerin (Seiler-Ausbildungsverordnung -SeilAusbV),2008-05-31T00:00:00Z
2,Verordnung über das Meisterprüfungsberufsbild und über die Prüfungsanforderungen in den Teilen I und II der Meisterp...,2008-06-30T00:00:00Z
3,Verordnung über das Meisterprüfungsberufsbild und über die Prüfungsanforderungen in den Teilen I und II der Meisterp...,2008-06-30T00:00:00Z
4,Verordnung über das Meisterprüfungsberufsbild und über die Prüfungsanforderungen in den Teilen I und II der Meisterp...,2009-03-11T00:00:00Z
5,Verordnung über das Meisterprüfungsberufsbild und über die Prüfungsanforderungen in den Teilen I und II der Meisterp...,2009-09-24T00:00:00Z
6,Verordnung über die Berufsausbildung zum Milchtechnologen/zur Milchtechnologin,2010-04-16T00:00:00Z
7,Verordnung über die Berufsausbildung zum Böttcher und zur Böttcherin (Böttchergewerbe-Ausbildungsverordnung  Böttch...,2010-05-18T00:00:00Z
8,Verordnung über die Berufsausbildung zum Büchsenmacher und zur Büchsenmacherin (Büchsenmacher-Ausbildungsverordnung ...,2010-06-02T00:00:00Z
9,Verordnung über die Meisterprüfung zum anerkannten Fortbildungsabschluss Agrarservicemeister und Agrarservicemeister...,2010-08-23T00:00:00Z



── topic 15  (n=61)  top words: nderung steuerlich, steuerlich, prfungsordnung, einkommensteuergesetz, vermgen, verhinderung


,title,date_published
0,Verordnung zur Änderung von Verbrauchsteuerverordnungen sowie der Brennereiordnung,2008-03-31T00:00:00Z
1,Achtes Gesetz zur Änderung des Steuerberatungsgesetzes,2008-04-11T00:00:00Z
2,Gesetz zur Modernisierung und Entbürokratisierung des Steuerverfahrens (Steuerbürokratieabbaugesetz),2008-12-24T00:00:00Z
3,Jahressteuergesetz 2009 (JStG 2009),2008-12-24T00:00:00Z
4,Gesetz zur Reform des Erbschaftsteuer- und Bewertungsrechts (Erbschaftsteuerreformgesetz - ErbStRG),2008-12-31T00:00:00Z
5,Viertes Gesetz zur Änderung von Verbrauchsteuergesetzen,2009-07-21T00:00:00Z
6,Gesetz zur Bekämpfung der Steuerhinterziehung (Steuerhinterziehungsbekämpfungsgesetz),2009-07-31T00:00:00Z
7,Steuerhinterziehungsbekämpfungsverordnung (SteuerHBekV),2009-09-24T00:00:00Z
8,Bekanntmachung über das Inkrafttreten von Teilen des Vierten Gesetzes zur Änderung von Verbrauchsteuergesetzen,2009-10-02T00:00:00Z
9,Fünfte Verordnung zur Änderung von Verbrauchsteuerverordnungen,2009-10-09T00:00:00Z



── topic 16  (n=26)  top words: beschftigung, arbeitnehmerentsendegesetz, arbeitszeit, arbeitnehmer, instrument, verbesserung international


...

,title,date_published
0,Achte Verordnung zur Änderung der Ferienreiseverordnung,2008-06-19T00:00:00Z
1,Zwölfte Verordnung zur Änderung der Erholungsurlaubsverordnung,2009-07-24T00:00:00Z
2,Zweite Verordnung zur Änderung der Heimaturlaubsverordnung,2010-06-18T00:00:00Z
3,Neunte Verordnung zur Änderung der Ferienreiseverordnung,2012-06-22T00:00:00Z
4,Dreizehnte Verordnung zur Änderung der Erholungsurlaubsverordnung,2012-12-13T00:00:00Z
5,Zehnte Verordnung zur Änderung der Ferienreiseverordnung,2013-06-20T00:00:00Z
6,Vierzehnte Verordnung zur Änderung der Erholungsurlaubsverordnung,2014-11-28T00:00:00Z
7,Elfte Verordnung zur Änderung der Ferienreiseverordnung,2015-06-29T00:00:00Z
8,Erste Verordnung zur Änderung der Sonderurlaubsverordnung,2021-06-08T00:00:00Z
9,Zweite Verordnung zur Änderung der Sonderurlaubsverordnung,2021-12-29T00:00:00Z



── topic 131  (n=8)  top words: bundesbesoldungsgesetz, 77, 78, 5a, anlage iv, iv


,title,date_published
0,Bekanntmachung nach § 77 Abs. 2 und 3 des Bundesbesoldungsgesetzes und nach § 2 Abs. 1 und § 3 Abs. 2 der Zweiten Be...,2008-08-01T00:00:00Z
1,Bekanntmachung nach § 77 Absatz 1 und 2 sowie nach § 78 Absatz 1 des Bundesbesoldungsgesetzes und nach § 6 Absatz 2 ...,2009-07-02T00:00:00Z
2,Berichtigung der Bekanntmachung nach § 77 Absatz 1 und 2 sowie nach § 78 Absatz 1 des Bundesbesoldungsgesetzes und n...,2009-08-19T00:00:00Z
3,Bekanntmachung nach § 77 Absatz 4 und § 78 Absatz 2 des Bundesbesoldungsgesetzes sowie nach § 6 Absatz 3 des Besoldu...,2012-08-23T00:00:00Z
4,Bekanntmachung nach § 77 Absatz 4 und § 78 Absatz 2 des Bundesbesoldungsgesetzes sowie nach § 5a Absatz 2 und § 6 Ab...,2014-12-04T00:00:00Z
5,Bekanntmachung nach § 78 Absatz 2 des Bundesbesoldungsgesetzes,2019-12-20T00:00:00Z
6,Bekanntmachung nach Anlage IV Nummer 1 und 4 des Bundesbesoldungsgesetzes,2020-03-05T00:00:00Z
7,"Bekanntmachung nach § 77 Absatz 4, § 78 Absatz 2 und Anlage IV Nummer 1 und 4 des Bundesbesoldungsgesetzes sowie nac...",2021-08-13T00:00:00Z



── topic 133  (n=6)  top words: ii, trger, rente, bundesagentur arbeit, zahlung, bundesagentur


,title,date_published
0,Erste Verordnung zur Änderung der Arbeitslosengeld II/Sozialgeld-Verordnung,2008-12-23T00:00:00Z
1,Dritte Verordnung zur Änderung der Arbeitslosengeld II/Sozialgeld-Verordnung,2010-05-12T00:00:00Z
2,Erste Verordnung zur Änderung der Verordnung über die Pauschalierung und Zahlung des Ausgleichsbetrags der Bundesage...,2010-12-22T00:00:00Z
3,Fünfte Verordnung zur Änderung der Arbeitslosengeld II/Sozialgeld-Verordnung,2011-06-25T00:00:00Z
4,Neunte Verordnung zur Änderung der Arbeitslosengeld II/Sozialgeld-Verordnung,2020-12-16T00:00:00Z
5,Zehnte Verordnung zur Änderung der Arbeitslosengeld II/Sozialgeld-Verordnung,2021-03-24T00:00:00Z


### §6a. Decision — System 2 only (topic level)

Review the topic groups above. Enter the **topic IDs** you want to keep in `S2_KEPT_TOPICS` below — all documents in those topics are included, all others are excluded.  
Leave as `[]` to exclude all System-2-only policies.

In [ ]:
# ── DECISION: System 2 only ───────────────────────────────────────────────────
# BERTopic topic IDs to keep in full (all documents in those topics are included).
# Topic IDs are shown in the "── topic N" headers above.
# Example: S2_KEPT_TOPICS = [3, 7, 12]
# Leave as [] to exclude all System-2-only policies.

S2_KEPT_TOPICS = [7, 16, 17, 20, 39, 42, 86, 100, 109, 121, 122, 125, 131, 133]   # ← edit this list

# Individual picks from topics NOT fully kept above.
# Keys: topic ID (from §6 "── topic N" header).
# Values: 0-based row indices within that topic's table (left-most column in §6 display).
# Example: S2_INDIVIDUAL_PICKS = {5: [0, 3], 14: [2]}

S2_INDIVIDUAL_PICKS = {15: [10, 11], 44: [7, 11], 46: [6, 7, 15], 51: [4, 10, 9], 
                       54: [4, 5, 6, 8, 11, 18, 28, 29], 76: [3, 5]}   # ← edit this dict

## §7. Noise pool spot-check

Topic -1: documents HDBSCAN could not assign to any cluster. Spot-check a sample
to verify they are correctly excluded as non-welfare-policy laws.

In [ ]:
SAMPLE_N = 30
if noise_docs:
    sample   = random.sample(noise_docs, min(SAMPLE_N, len(noise_docs)))
    df_noise = pd.DataFrame(sample)
    cols     = [c for c in ['title', 'date_published'] if c in df_noise.columns]
    df_noise = df_noise[cols].sort_values('date_published').reset_index(drop=True)
    print(f'Random sample of {len(df_noise)} from {len(noise_docs)} noise docs (topic -1)')
    display(df_noise)
else:
    print('No noise pool file found.')

---

*Add further inspection cells below.*

## §8. Final policy set — compile & save

Applies the decisions from §5a and §6a.  
All "both" candidates are always included; System-1-only and System-2-only policies are filtered based on your selections above.

In [ ]:
# ── Apply decisions ───────────────────────────────────────────────────────────
df_both_docs   = df_union[df_union['system_sources'] == 'both'].copy()
df_s1_selected = df_s1.iloc[S1_INCLUDED_INDICES].copy() if S1_INCLUDED_INDICES else df_s1.iloc[[]]
df_s2_selected = (
    df_s2[df_s2['berttopic_topic'].isin(S2_KEPT_TOPICS)].copy()
    if S2_KEPT_TOPICS else df_s2.iloc[[]]
)

# Individual picks from topics not fully kept
_pick_frames = []
for _topic_id, _indices in S2_INDIVIDUAL_PICKS.items():
    if _topic_id in S2_KEPT_TOPICS:
        continue  # already fully included
    _group = df_s2[df_s2['berttopic_topic'] == _topic_id].reset_index(drop=True)
    _pick_frames.append(_group.iloc[_indices])
df_s2_individual = pd.concat(_pick_frames, ignore_index=True) if _pick_frames else df_s2.iloc[[]]

keep_titles = (
    set(df_both_docs['title'].str.strip())
    | set(df_s1_selected['title'].str.strip())
    | set(df_s2_selected['title'].str.strip())
    | set(df_s2_individual['title'].str.strip())
)

final_records = [r for r in union if r['title'].strip() in keep_titles]
final_records.sort(key=lambda x: x.get('date_published', ''))

FINAL_FILE = BASE / f'{OUTPUT_PREFIX}_final_policy_set.json'
FINAL_FILE.write_text(json.dumps(final_records, ensure_ascii=False, indent=2), encoding='utf-8')

n_both_final       = len(df_both_docs)
n_s1_final         = len(df_s1_selected)
n_s2_final         = len(df_s2_selected)
n_s2_individual    = len(df_s2_individual)
n_total_final      = len(final_records)

print(f'Accepted from both systems:           {n_both_final:>4}')
print(f'Accepted from System 1 only:          {n_s1_final:>4}  (of {n_s1} candidates)')
print(f'Accepted from System 2 only (topics): {n_s2_final:>4}  (of {n_s2} candidates, {len(S2_KEPT_TOPICS)} topic(s) kept)')
print(f'Accepted from System 2 (individual):  {n_s2_individual:>4}  (across {len(S2_INDIVIDUAL_PICKS)} topic(s))')
print(f'─────────────────────────────────────────────')
print(f'Final policy set (total):             {n_total_final:>4}')
print(f'Saved to: {FINAL_FILE.name}')

## §9. Export evaluation log

Generates a dated Markdown log documenting all inputs, counts, decisions, and outputs.  
**Run after §8.**

In [ ]:
import datetime

today = datetime.date.today().isoformat()
log_path = BASE / f'pooled_eval_log_{today}.md'

s2_topics_detail = ', '.join(
    f'topic {t} ({", ".join(str(w) for w in topic_to_words.get(t, [])[:5])})'
    for t in sorted(S2_KEPT_TOPICS)
) if S2_KEPT_TOPICS else '*(none)*'

log_md = f"""# Pooled Union Evaluation Log
Generated: {today}

## 1. Input files

| role | file |
|---|---|
| System 1 (cosine similarity) | `{S1_FILE.name}` |
| System 2 (BERTopic candidates) | `{S2_FILE.name}` |
| System 2 noise pool | `{NOISE_FILE.name}` |
| BERTopic topic assignments | `{TOPIC_ASSIGN_FILE.name}` |
| BERTopic topic info | `{TOPIC_INFO_FILE.name}` |

## 2. §1 — Candidate counts (raw input)

| source | n |
|---|---|
| System 1 (cosine similarity) | {len(s1_docs)} |
| System 2 (BERTopic) | {len(s2_docs)} |
| System 2 noise pool (topic -1) | {len(noise_docs)} |

## 3. §3 — Pooled union overview

| group | n | share |
|---|---|---|
| Both systems (high confidence) | {n_both} | {n_both/n_total:.0%} |
| System 1 only (cosine sim) | {n_s1} | {n_s1/n_total:.0%} |
| System 2 only (BERTopic) | {n_s2} | {n_s2/n_total:.0%} |
| **Total** | **{n_total}** | **100%** |

## 4. §5a — System 1 only decision

- Candidates reviewed: {n_s1}
- Row indices included: {S1_INCLUDED_INDICES if S1_INCLUDED_INDICES else '(none)'}
- **Policies included: {n_s1_final}**

## 5. §6a — System 2 only decision (topic level)

- Candidates reviewed: {n_s2} across {df_s2['berttopic_topic'].nunique()} topics
- Topics kept: {len(S2_KEPT_TOPICS)} — {s2_topics_detail}
- **Policies included: {n_s2_final}**

## 6. §8 — Final policy set

| group | n |
|---|---|
| Accepted by both systems | {n_both_final} |
| Accepted from System 1 only | {n_s1_final} |
| Accepted from System 2 only | {n_s2_final} |
| **Final total** | **{n_total_final}** |

## 7. Output file

`{FINAL_FILE.name}`
"""

log_path.write_text(log_md, encoding='utf-8')
print(f'Log saved to: {log_path.name}')